In [25]:
import os.path as op
import os
import numpy as np
import nibabel as nb
from ipyparallel import Client

In [26]:
engines = Client()

In [27]:
home_dir =  '/home/dnl/habitization/'
data_dir = home_dir + '/data/'
subs = home_dir + 'subjects.txt'
subs = list(np.loadtxt(subs,str))
TEST = 'SV'
VARS = ['1','2']

new_folder = home_dir + 'templates'+'/glms_' + TEST + '_step3/'

sessions = ['a','b','meandiff']

ignore_subs = []#'HAB02', 'HAB03', 'HAB05', 'HAB08', 'HAB10', 'HAB11', 'HAB12', 'HAB13', 'HAB14', 'HAB16','HAB01', 'HAB18', 'HAB19', 'HAB20', 'HAB21', 'HAB22', 'HAB23', 'HAB24', 'HAB25']
#First analysis included subjects 'HAB02', 'HAB03', 'HAB05', 'HAB08', 'HAB10', 'HAB11', 'HAB12', 'HAB13', 'HAB14', 'HAB16'
#Second analysis included subjects 'HAB01', 'HAB07', 'HAB17', 'HAB18', 'HAB19', 'HAB20', 'HAB21', 'HAB22', 'HAB23', 'HAB24', 'HAB25'
subs = [x for x in subs if x not in ignore_subs]
print(subs)

['HAB01', 'HAB02', 'HAB03', 'HAB05', 'HAB06', 'HAB07', 'HAB08', 'HAB10', 'HAB11', 'HAB12', 'HAB13', 'HAB14', 'HAB16', 'HAB18', 'HAB19', 'HAB20', 'HAB21', 'HAB22', 'HAB23', 'HAB24', 'HAB25']


In [28]:
def make_new_dir(dir_name):
    if not op.exists(dir_name):
        os.mkdir(dir_name)

In [29]:
#finds and replaces info in text file that writes analysis
for VAR in VARS:
    for sesh in sessions:
        
        if sesh == 'meandiff':
            template = op.join(home_dir,'templates','glm_1variable_step3meandiff.fsf')        
        
        else:
            template = op.join(home_dir,'templates','glm_1variable_step3.fsf')
        
        
        new_f = op.join(home_dir,'templates','glms_' + TEST + '_step3','_'.join([VAR,sesh]) +'.fsf')
            
        make_new_dir(new_folder)
            
        with open(template,'r') as file:
            filedata = file.read()
                       
        filedata = filedata.replace('$SESSION',sesh)
        filedata = filedata.replace('$TEST',TEST)
        filedata = filedata.replace('$VAR',VAR)
            
            #next get number of time points
            #func = op.join(home_dir,'data',sub,'func', sesh + run + '.feat','filtered_func_data.nii.gz')
            #func = nb.load(func)
            #ntpts = func.shape[-1]
            #filedata = filedata.replace('$NUM_TIMEPOINTS',str(ntpts))

            
        with open(new_f,'w') as file:
            file.write(filedata)


In [30]:
def feat(in_tuple):
    VAR,sesh = in_tuple
    fsf = op.join(new_folder,'_'.join([VAR,sesh]) +'.fsf')
    
    cmd = ['feat',fsf]
    cmd = ' '.join(cmd)
    os.system(cmd)
    return in_tuple

In [31]:
in_tuples = []

for VAR in VARS:
    for sesh in sessions:
        in_tuples.append((VAR,sesh))

In [32]:
sub_engines = engines[0:24] #less if you want to use fewer
sub_engines.push(dict(new_folder = new_folder))
sub_engines.execute('import os.path as op')
with sub_engines.sync_imports():
    import os
    
output = sub_engines.map_sync(feat,in_tuples)

importing os on engine(s)




print('finished')